# Local Secure Dedup Demo Walkthrough

This notebook is designed for complete reviewer transparency.

It keeps everything visible in one place:

- the current backend status of the live API,
- the collected test cases,
- the full test source files,
- the saved test reports and raw outputs,
- optional rerun cells for the main suites,
- the benchmark script source and comparison artifacts,
- the live PoW / chunk-sharing / rate-limit demo,
- the dataset and model metrics behind the behavioural layer.

Backend note:

- The repo supports `Redis` for index / policy / reputation state, with in-memory fallback.
- The repo supports `LocalStack` / S3-style chunk storage, with filesystem fallback.
- The first cells below show what the currently running server is actually using.

In [ ]:
import json
import subprocess
import sys
import tempfile
import time
from html import escape
from pathlib import Path

import pandas as pd
import requests
from IPython.display import HTML, Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'app.py').exists() and (candidate / 'docs').exists():
            return candidate
    raise RuntimeError('Could not find repo root from the notebook working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())

def choose_repo_python() -> str:
    candidates = [
        REPO_ROOT / '.venv' / 'Scripts' / 'python.exe',
        REPO_ROOT / '.venv' / 'bin' / 'python',
        Path(sys.executable),
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            return str(candidate)
    return sys.executable

REPO_PYTHON = choose_repo_python()
BASE_URL = 'http://127.0.0.1:8000'
API_KEY = 'dev-api-key'
CLIENT_ID = f'notebook-demo-{int(time.time())}'
HEADERS = {'X-API-Key': API_KEY, 'X-Client-ID': CLIENT_ID}

def run_and_show(args, cwd=REPO_ROOT):
    print('COMMAND:', ' '.join(str(arg) for arg in args))
    result = subprocess.run(args, cwd=str(cwd), capture_output=True, text=True, check=False)
    print('RETURN CODE:', result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    return result

def show_text_file(path: Path, language: str = 'text'):
    text = path.read_text(encoding='utf-8')
    numbered = '\n'.join(f'{idx:4}: {line}' for idx, line in enumerate(text.splitlines(), start=1))
    rel = path.relative_to(REPO_ROOT)
    display(HTML(f'<h4>{escape(str(rel))}</h4><pre>{escape(numbered)}</pre>'))

print('REPO_ROOT   =', REPO_ROOT)
print('REPO_PYTHON =', REPO_PYTHON)
print('BASE_URL    =', BASE_URL)
print('CLIENT_ID   =', CLIENT_ID)

In [ ]:
print('Docker compose status:')
run_and_show(['docker', 'compose', '-f', str(REPO_ROOT / 'docker-compose.local.yml'), 'ps'])

health = requests.get(f'{BASE_URL}/health', timeout=30)
health.raise_for_status()
config = requests.get(f'{BASE_URL}/demo/config', timeout=30)
config.raise_for_status()
config = config.json()

summary = {
    'health': health.json(),
    'demo_mode': config['demo_mode'],
    'storage_backend': config['storage']['backend'],
    'configured_storage_backend': config['storage']['configured_backend'],
    'storage_fallback_active': config['storage']['fallback_active'],
    'storage_endpoint': config['storage']['endpoint'],
    'storage_bucket': config['storage']['bucket'],
    'fingerprint_mode': config['fingerprint']['mode'],
    'detection_mode': config['detection']['mode'],
}
print(json.dumps(summary, indent=2))

print('\nLocalStack bucket visibility:')
run_and_show(['docker', 'exec', 'secure_dedup_localstack', 'awslocal', 's3', 'ls'])

print('Redis visibility:')
run_and_show(['docker', 'exec', 'secure_dedup_redis', 'redis-cli', 'ping'])

## Saved Test Reports

These are the persisted human-readable reports from the latest saved pytest runs.

In [ ]:
display(Markdown((REPO_ROOT / 'test_reports' / 'frequency_attack_pytest_20260322_181202.md').read_text(encoding='utf-8')))
display(Markdown((REPO_ROOT / 'test_reports' / 'attack_detection_demo_pytest_20260322_181200.md').read_text(encoding='utf-8')))

## Collected Test Cases

This cell lists the individual tests so reviewers can see the exact case names before looking at the source.

In [ ]:
run_and_show([
    REPO_PYTHON,
    '-m',
    'pytest',
    'tests/test_frequency_attack_resistance.py',
    'tests/test_attack_detection_demo.py',
    'tests/test_encryption.py',
    '--collect-only',
    '-q',
])

## Full Test Source: Frequency Attack Resistance

In [ ]:
show_text_file(REPO_ROOT / 'tests' / 'test_frequency_attack_resistance.py', language='python')

## Full Test Source: Behavioural Attack Detection

In [ ]:
show_text_file(REPO_ROOT / 'tests' / 'test_attack_detection_demo.py', language='python')

## Full Test Source: Encryption Unit Tests

In [ ]:
show_text_file(REPO_ROOT / 'tests' / 'test_encryption.py', language='python')

## Saved Raw Pytest Output

These are the raw saved outputs, not paraphrased summaries.

In [ ]:
show_text_file(REPO_ROOT / 'test_reports' / 'frequency_attack_pytest_20260322_181202.txt')
show_text_file(REPO_ROOT / 'test_reports' / 'attack_detection_demo_pytest_20260322_181200.txt')

## Optional Transparent Reruns

These cells rerun the main suites directly from the notebook.

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_frequency_attack_resistance.py', '-v', '-s'])

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_attack_detection_demo.py', '-v', '-s'])

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_encryption.py', '-q'])

## Benchmark Script Transparency

This is the actual script that generated the encryption comparison artifact.

In [ ]:
show_text_file(REPO_ROOT / 'compare_dedup_encryption_schemes.py', language='python')

In [ ]:
run_and_show([REPO_PYTHON, 'compare_dedup_encryption_schemes.py', '--print-table'])

In [ ]:
comparison = json.loads((REPO_ROOT / 'docs' / 'project_notes' / 'encryption_scheme_comparison.json').read_text(encoding='utf-8'))

comparison_rows = []
for scheme in comparison['schemes']:
    comparison_rows.append({
        'scheme': scheme['scheme'],
        'dedup_saved_percent': scheme['dedup_saved_percent'],
        'avg_token_time_ms': scheme['avg_token_time_ms'],
        'avg_encrypt_time_ms': scheme['avg_encrypt_time_ms'],
        'avg_decrypt_time_ms': scheme['avg_decrypt_time_ms'],
        'token_reproducible_without_secret': scheme['security_properties']['token_reproducible_without_secret'],
        'frequency_attack_resistant': scheme['security_properties']['frequency_attack_resistant'],
        'external_key_server_required': scheme['security_properties']['external_key_server_required'],
    })

display(pd.DataFrame(comparison_rows))
print('Delta summary:')
print(json.dumps(comparison['comparison'], indent=2))

## Live API Demo: Controlled Similar Files, PoW, and Shared Chunks

The file construction below deliberately keeps two large common regions with different middle regions, so shared chunks are visible and repeatable.

In [ ]:
CHUNK_SIZE = 16384

def build_block(tag: str, size: int = CHUNK_SIZE) -> bytes:
    raw = (tag * ((size // len(tag)) + 2)).encode('ascii')
    return raw[:size]

def upload_with_optional_pow(path: Path, pow_proofs_json=None):
    data = {}
    if pow_proofs_json is not None:
        data['pow_proofs_json'] = json.dumps(pow_proofs_json)
    with path.open('rb') as fh:
        response = requests.post(
            f'{BASE_URL}/upload',
            headers=HEADERS,
            files={'file': (path.name, fh, 'text/plain')},
            data=data,
            timeout=180,
        )
    return response

def solve_pow(challenges):
    payload = {
        'challenges': [
            {
                'chunk_hash': item['chunk_hash'],
                'challenge_id': item['challenge_id'],
                'nonce_hex': item['nonce_hex'],
                'offset': item['offset'],
                'length': item['length'],
            }
            for item in challenges
        ]
    }
    response = requests.post(
        f'{BASE_URL}/demo/solve_pow',
        headers={'X-API-Key': API_KEY},
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    return response.json()['pow_proofs']

session = str(int(time.time()))
common_left = build_block(f'COMMON-LEFT-{session}-')
variant_a = build_block(f'VARIANT-A-{session}-')
variant_b = build_block(f'VARIANT-B-{session}-')
common_right = build_block(f'COMMON-RIGHT-{session}-')

file_a_bytes = common_left + variant_a + common_right
file_b_bytes = common_left + variant_b + common_right

work = Path(tempfile.mkdtemp(prefix='secure-dedup-notebook-'))
path_a = work / 'similar_a.txt'
path_b = work / 'similar_b.txt'
path_a.write_bytes(file_a_bytes)
path_b.write_bytes(file_b_bytes)

upload_a = upload_with_optional_pow(path_a)
upload_a.raise_for_status()
body_a = upload_a.json()

upload_b_first = upload_with_optional_pow(path_b)
body_b_first = upload_b_first.json()

if upload_b_first.status_code == 409:
    proofs = solve_pow(body_b_first['detail']['required_challenges'])
    upload_b = upload_with_optional_pow(path_b, pow_proofs_json=proofs)
else:
    proofs = {}
    upload_b = upload_b_first

upload_b.raise_for_status()
body_b = upload_b.json()

compare = requests.get(
    f'{BASE_URL}/demo/compare-files',
    headers=HEADERS,
    params={
        'file_id_a': body_a['file']['file_id'],
        'file_id_b': body_b['file']['file_id'],
    },
    timeout=180,
)
compare.raise_for_status()
compare_body = compare.json()['comparison']

print('Upload A chunk summary:')
print(json.dumps(body_a['chunk_summary'], indent=2))
print('\nUpload B first response status:', upload_b_first.status_code)
print(json.dumps(body_b_first, indent=2))
print('\nUpload B final chunk summary:')
print(json.dumps(body_b['chunk_summary'], indent=2))
print('\nComputed PoW proofs:')
print(json.dumps(proofs, indent=2))

shared_positions_df = pd.DataFrame(compare_body['shared_chunk_positions'])
display(shared_positions_df)
print('\nShared chunk count =', compare_body['shared_chunk_count'])
print('Interpretation =', compare_body['interpretation'])

## Live API Demo: Forced Rate Limit and Highlights

In [ ]:
force = requests.post(
    f'{BASE_URL}/demo/force-policy',
    headers={'X-API-Key': API_KEY},
    json={'client_id': CLIENT_ID, 'action': 'RATE_LIMIT'},
    timeout=60,
)
force.raise_for_status()

attack_path = work / 'attack_check.txt'
attack_path.write_text('attack-check-' + session, encoding='utf-8')
with attack_path.open('rb') as fh:
    blocked = requests.post(
        f'{BASE_URL}/upload',
        headers=HEADERS,
        files={'file': (attack_path.name, fh, 'text/plain')},
        timeout=180,
    )

blocked_body = blocked.json()
highlights = requests.get(
    f'{BASE_URL}/demo/highlights/{CLIENT_ID}',
    headers={'X-API-Key': API_KEY},
    timeout=180,
)
highlights.raise_for_status()
highlights_body = highlights.json()

clear = requests.post(
    f'{BASE_URL}/demo/clear-policy',
    headers={'X-API-Key': API_KEY},
    json={'client_id': CLIENT_ID},
    timeout=60,
)
clear.raise_for_status()

print('Forced policy response:')
print(json.dumps(force.json(), indent=2))
print('\nBlocked upload status:', blocked.status_code)
print(json.dumps(blocked_body, indent=2))
print('\nHighlights summary:')
print(json.dumps(highlights_body['highlights'], indent=2))
display(pd.DataFrame(highlights_body['recent_events']))

## Dataset and Model Transparency

In [ ]:
training_metrics = json.loads((REPO_ROOT / 'dense_artifacts' / 'training_metrics.json').read_text(encoding='utf-8'))
summary = {
    'rows': training_metrics['rows'],
    'train_rows': training_metrics['train_rows'],
    'test_rows': training_metrics['test_rows'],
    'best_model': training_metrics['best_model'],
    'best_cv_score': training_metrics['best_cv_score'],
}
print(json.dumps(summary, indent=2))

class_distribution = pd.DataFrame(
    list(training_metrics['class_distribution'].items()),
    columns=['attack_label', 'rows'],
).sort_values('rows', ascending=False)
display(class_distribution)

candidate_results = pd.DataFrame(training_metrics['candidate_results']).sort_values('best_cv_score', ascending=False)
display(candidate_results[['model', 'best_cv_score']])

In [ ]:
display(Markdown((REPO_ROOT / 'dense_artifacts' / 'evaluation_report.md').read_text(encoding='utf-8')))

## Reviewer Summary

- The notebook shows the live backend state first, so the infrastructure claim is visible.
- The collected pytest names, full source files, saved reports, and raw outputs are all shown directly.
- The rerun cells allow the reviewers to watch the tests execute, not just read a report.
- The benchmark code, benchmark output, and benchmark artifact are all visible.
- The live API flow shows aligned shared chunks, PoW, and rate limiting with LocalStack-backed storage.